In [ ]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobServiceClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType, SimpleField, SearchableField,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SearchIndexerDataSourceConnection, SearchIndexerDataContainer,
    AzureOpenAIEmbeddingSkill, InputFieldMappingEntry, OutputFieldMappingEntry, 
    SearchIndexerSkillset, SearchIndexer, FieldMapping,
)

# --- CONFIGURATION ---
AZURE_OPENAI_ENDPOINT = "https://temp-m2web-ai-search-pr-resource.cognitiveservices.azure.com/"
AZURE_OPENAI_KEY = "<REDACTED_API_KEY>"

SEARCH_ENDPOINT = "https://rush-lyric-ai-search.search.windows.net"
SEARCH_KEY = "<REDACTED_API_KEY>" # Full 52 chars

STORAGE_CONNECTION_STRING = "<REDACTED_CONNECTION_STRING>" # Paste your full DefaultEndpointsProtocol string here
# ---------------------

In [ ]:
# 1. Configure Vectorizer
vectorizer = AzureOpenAIVectorizer(
    vectorizer_name="myVectorizer",
    parameters=AzureOpenAIVectorizerParameters(
        resource_url="https://temp-m2web-ai-search-pr-resource.cognitiveservices.azure.com/",
        deployment_name="text-embedding-3-small",
        model_name="text-embedding-3-small",
        api_key="<REDACTED_API_KEY>"
    )
)

Index 'integrated-index' created successfully.


In [3]:
# create the container
container_name = "rush-lyrics-data"
local_file_path = "rush_lyrics_for_indexing.jsonl"

blob_service_client = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(container_name)

if not container_client.exists():
    container_client.create_container()

with open(local_file_path, "rb") as data:
    container_client.upload_blob(name="rush_lyrics.jsonl", data=data, overwrite=True)
print(f"File '{local_file_path}' uploaded to container '{container_name}'.")

File 'rush_lyrics_for_indexing.jsonl' uploaded to container 'rush-lyrics-data'.


In [6]:
blobs = container_client.list_blobs()
for blob in blobs:
    print(f"Found: {blob.name}")

Found: rush_lyrics.jsonl
Found: rush_lyrics2.jsonl


In [ ]:
from azure.storage.blob import BlobServiceClient

# 1. Connect and Define Names
# IMPORTANT: Copy the "Connection string", NOT the "Key".
# ![image](images/34/azure-storage-keys-final.png)
connection_string = "<REDACTED_CONNECTION_STRING>"
container_name = "rush-lyrics-data"
# NOTE: Azure ML Compute Instances are LINUX. Do not use C:\ paths.
# If the notebook is in the same folder as the file:
local_file_path = "rush_lyrics_for_indexing.jsonl"

# To find your absolute Linux path, run: !pwd in a notebook cell
# Example: local_file_path = "/home/azureuser/cloudfiles/code/.../rush_lyrics_for_indexing.jsonl"

# 2. Upload to Cloud
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

# Create container if it doesn't exist
if not container_client.exists():
    container_client.create_container()

with open(local_file_path, "rb") as data:
    container_client.upload_blob(name="rush_lyrics2.jsonl", data=data, overwrite=True)

In [7]:
# 1. Get the blob client for the file you want to remove
old_blob_client = container_client.get_blob_client("rush_lyrics.jsonl")

# 2. Delete the blob
if old_blob_client.exists():
    old_blob_client.delete_blob()
    print("Deleted: rush_lyrics.jsonl")
else:
    print("File not found or already deleted.")

# 3. Verify what remains
print("\nRemaining files in container:")
blobs = container_client.list_blobs()
for blob in blobs:
    print(f"Found: {blob.name}")

Deleted: rush_lyrics.jsonl

Remaining files in container:
Found: rush_lyrics2.jsonl


In [8]:
# 1. Define source and destination names
source_blob_name = "rush_lyrics2.jsonl"
dest_blob_name = "rush_lyrics.jsonl"

# 2. Get the specific blob clients
source_blob_client = container_client.get_blob_client(source_blob_name)
dest_blob_client = container_client.get_blob_client(dest_blob_name)

# 3. Copy the data (this will overwrite the old rush_lyrics.jsonl)
dest_blob_client.start_copy_from_url(source_blob_client.url)

# 4. Delete the original rush_lyrics2.jsonl
source_blob_client.delete_blob()

print(f"Successfully renamed {source_blob_name} to {dest_blob_name}")

# Verify the result
print("\nCurrent files in container:")
for blob in container_client.list_blobs():
    print(f"Found: {blob.name}")

Successfully renamed rush_lyrics2.jsonl to rush_lyrics.jsonl

Current files in container:
Found: rush_lyrics.jsonl
